# C5. Prior work check on arXiv

The application says this would be the first PQK classification of genomic data above 61 qubits. This notebook
queries the arXiv API, flags papers that report large qubit counts, use projected kernels, or touch biomedical
data, and lists the ones to read before you submit. It complements a manual search; it does not replace one.

Needs internet. Takes about a minute, because the API asks for 3 s between calls.

In [1]:
SINCE   = "2025-01-01"     # submission date cutoff
MAX_PER = 200              # results per query
QUERIES = [
    'abs:"projected quantum kernel"',
    'abs:"projected quantum kernels"',
    'abs:"projected kernel" AND abs:quantum',
    'abs:"quantum kernel" AND abs:genomic',
    'abs:"quantum kernel" AND abs:"gene expression"',
    'abs:"quantum kernel" AND abs:cancer',
    'abs:"quantum machine learning" AND abs:"gene expression"',
    'abs:"quantum kernel" AND abs:"utility scale"',
    'abs:"quantum kernel" AND abs:"156 qubits"',
    'abs:"quantum kernel" AND abs:"133 qubits"',
    'abs:"quantum kernel" AND abs:"127 qubits"',
    'abs:"quantum kernel" AND abs:"100 qubits"',
]
KNOWN = {"2507.22710": "IBM, 61-qubit PQK, CAR T-cell", "2605.24252": "100-qubit PQK, time series",
         "2412.07915": "156-qubit fidelity kernels"}

In [2]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
OUT = ROOT / "credits" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import re, urllib.parse, urllib.request
import xml.etree.ElementTree as ET

project folder: D:\Active Research Projects\PQK100


## C5.1 Query

In [3]:
NS = {"a": "http://www.w3.org/2005/Atom"}
API = "https://export.arxiv.org/api/query?"

def query(q):
    url = API + urllib.parse.urlencode({"search_query": q, "start": 0, "max_results": MAX_PER,
                                        "sortBy": "submittedDate", "sortOrder": "descending"})
    for wait in (20, 60, 120, None):
        try:
            with urllib.request.urlopen(url, timeout=45) as r:
                root = ET.fromstring(r.read())
            break
        except Exception as e:
            why = getattr(e, "code", type(e).__name__)
            if wait is None:
                print(f"  gave up on this query ({why})"); return None
            print(f"  arXiv said {why}; waiting {wait} s"); time.sleep(wait)
    out = []
    for e in root.findall("a:entry", NS):
        aid = e.find("a:id", NS).text.rsplit("/", 1)[-1]
        out.append({"id": re.sub(r"v\d+$", "", aid), "date": e.find("a:published", NS).text[:10],
                    "title": " ".join(e.find("a:title", NS).text.split()),
                    "abstract": " ".join(e.find("a:summary", NS).text.split())})
    return out

hits, failed = {}, []
for q in QUERIES:
    res = query(q)
    if res is None:
        failed.append(q); continue
    print(f"{len(res):4d}  {q}")
    for r in res:
        hits.setdefault(r["id"], r)
    time.sleep(3.1)
if not hits:
    raise RuntimeError("arXiv returned nothing. Check the internet connection and run this cell again in a few minutes.")
papers = pd.DataFrame(hits.values())
print(f"\n{len(papers)} distinct papers" + (f"; {len(failed)} queries failed, rerun later: {failed}" if failed else ""))

  arXiv said 429; waiting 20 s
  arXiv said 429; waiting 60 s
  arXiv said 429; waiting 120 s
  gave up on this query (429)
  arXiv said 429; waiting 20 s
  arXiv said 429; waiting 60 s
  arXiv said 429; waiting 120 s
  gave up on this query (429)
  arXiv said 429; waiting 20 s
  arXiv said 429; waiting 60 s
  arXiv said 429; waiting 120 s
  gave up on this query (TimeoutError)
  arXiv said 429; waiting 20 s
  arXiv said 429; waiting 60 s
  arXiv said 429; waiting 120 s
  gave up on this query (429)
  arXiv said 429; waiting 20 s
  arXiv said 429; waiting 60 s
  arXiv said 429; waiting 120 s
  gave up on this query (429)
  arXiv said TimeoutError; waiting 20 s
  15  abs:"quantum kernel" AND abs:cancer
   5  abs:"quantum machine learning" AND abs:"gene expression"
   1  abs:"quantum kernel" AND abs:"utility scale"
   4  abs:"quantum kernel" AND abs:"156 qubits"
   1  abs:"quantum kernel" AND abs:"133 qubits"
   0  abs:"quantum kernel" AND abs:"127 qubits"
   2  abs:"quantum kernel" AND 

## C5.2 Flag

In [4]:
BIO = re.compile(r"gene|genom|transcript|omics|cancer|tumou?r|rna|protein|patient|clinical|biomed|cell", re.I)
PQK = re.compile(r"projected (quantum )?kernel", re.I)
QB  = re.compile(r"(\d{2,4})[\s-]+(?:physical\s+)?qubits?", re.I)

def max_qubits(text):
    ns = [int(m) for m in QB.findall(text)]
    return max(ns) if ns else None

papers["qubits"] = [max_qubits(r.title + " " + r.abstract) for r in papers.itertuples()]
papers["pqk"] = [bool(PQK.search(r.title + " " + r.abstract)) for r in papers.itertuples()]
papers["bio"] = [bool(BIO.search(r.title + " " + r.abstract)) for r in papers.itertuples()]
papers["link"] = "https://arxiv.org/abs/" + papers.id
recent = papers[papers.date >= SINCE].sort_values("date", ascending=False)
print(f"{len(recent)} papers since {SINCE}")
for k, v in KNOWN.items():
    print(f"  known {k} ({v}): {'found' if k in set(papers.id) else 'NOT found by these queries'}")

12 papers since 2025-01-01
  known 2507.22710 (IBM, 61-qubit PQK, CAR T-cell): NOT found by these queries
  known 2605.24252 (100-qubit PQK, time series): found
  known 2412.07915 (156-qubit fidelity kernels): found


## C5.3 Papers that could touch the "first" claim

Any paper that uses a quantum kernel above 61 qubits, or a projected kernel on biomedical data.
Read each abstract below. If one does PQK classification on genomic data above 61 qubits, the claim needs rewording.

In [5]:
big = recent.qubits.fillna(0) > 61
watch = recent[(big & (recent.pqk | recent.bio)) | (recent.pqk & recent.bio)]
cols = ["date", "id", "qubits", "pqk", "bio", "title"]
if watch.empty:
    print("No paper found that combines a quantum kernel above 61 qubits with PQK or biomedical data.")
else:
    display(watch[cols].reset_index(drop=True))
    for r in watch.itertuples():
        print(f"\n{r.date}  {r.link}\n{r.title}\n{r.abstract[:700]}...")

,date,id,qubits,pqk,bio,title
0,2026-05-22,2605.24252,114.0,True,False,Hybrid Quantum-Classical Machine Learning Algo...



2026-05-22  https://arxiv.org/abs/2605.24252
Hybrid Quantum-Classical Machine Learning Algorithms for Multi-Output Time-Series Forecasting at Utility Scale
Multi-output time-series forecasting in energy systems is challenging because of nonlinear dynamics, multi-scale seasonality, and strong dependencies across correlated series. In this work, we investigate two hybrid quantum-classical frameworks for multi-stream time-series forecasting on a real Smart Meter dataset comprising 103 household electricity consumption time-series, with experiments executed on the $ibm\_marrakesh$ superconducting quantum processor. The first model, Kernelized Quantum Reservoir Computing with Repeated Measurement (KQRC-RM), combines coupled quantum reservoirs, ancilla-assisted repeated measurement, and kernelized readouts to model temporal dynamics and cross-stream ...


In [6]:
recent[cols + ["link"]].to_csv(OUT / "arxiv_prior_work.csv", index=False)
print("all recent hits saved to", OUT / "arxiv_prior_work.csv")
recent[recent.pqk][cols].head(30).reset_index(drop=True)

all recent hits saved to D:\Active Research Projects\PQK100\credits\outputs\arxiv_prior_work.csv


,date,id,qubits,pqk,bio,title
0,2026-05-22,2605.24252,114.0,True,False,Hybrid Quantum-Classical Machine Learning Algo...
